In [2]:
import numpy as np
import torch

NumPy to PyTorch

- `torch.from_numpy(array)` converts a NumPy array into a PyTorch tensor
- NumPy's default decimal type is float64; PyTorch's is float32
- `from_numpy` keeps the NumPy dtype, so decimal arrays become float64 tensors
- Add `.type(torch.float32)` after converting, because models work in float32
- float64 data in a float32 model causes a "wrong datatype" error

Q1. Create a NumPy array holding 1.5, 2.5, 3.5. Convert it to a tensor. Predict the dtype first, then check.

Q2. Convert the same array into a float32 tensor in one line.

Q3. Create a random NumPy array with shape (2, 3) using np.random.rand(2, 3). Convert it to a tensor and print its shape and dtype.

Q4. Multiply your float64 tensor from Q1 by torch.ones(3), which is float32. What dtype does the result have? (Think about Topic 17: mixing dtypes gives the more precise type.)

In [3]:
array = np.array([1.5,2.5,3.5])
tensor_64 = torch.from_numpy(array)
tensor = torch.from_numpy(array).type(torch.float32)
print(tensor, tensor.dtype)
print(array, array.dtype)

random_array = np.random.rand(2,3)
tensor_array = torch.from_numpy(random_array)
print(tensor_array.shape, tensor_array.dtype)

ones = torch.ones(3)
result = tensor_64 * ones 
print(result.dtype)



tensor([1.5000, 2.5000, 3.5000]) torch.float32
[1.5 2.5 3.5] float64
torch.Size([2, 3]) torch.float64
torch.float64


PyTorch to NumPy

- `tensor.numpy()` converts a tensor into a NumPy array
- The dtype is kept: a float32 tensor becomes a float32 array
- `.numpy()` and `torch.from_numpy()` share data with the original (like a view)
- Reassigning (`x = x + 1`) creates a new object, so the other one does not change
- Changing in place (`x[0] = 100`) writes on the shared data, so both change
- Use `.clone()` before `.numpy()` for a fully independent copy
- `.numpy()` only works on CPU tensors (move GPU tensors back with `.cpu()` first)

Q1. Create a tensor of 5 zeros. Convert it to NumPy. Predict the dtype first, then print the array and its dtype.

Q2. After converting, reassign the tensor with + 5. Print the array. Did it change? Why?

Q3. Create a new tensor of 3 ones and convert it to NumPy. Then change the first value of the tensor to 50 in place. Print the array. Did it change? Why?

Q4. Redo Q3 so that the array does not change.

In [6]:
# Q1: tensor to NumPy keeps the dtype
t1 = torch.zeros(5)        # a float32 tensor of zeros
t2 = t1.numpy()            # convert: t2 shares data with t1
print(t1.dtype, t2.dtype)  # torch.float32 and float32

# Q2: reassigning does not affect the array
t1 = t1 + 5                # a NEW tensor; the name t1 moves to it
print(t2)                  # the array still holds the old zeros

# Q3: changing in place affects both
t3 = torch.ones(3)         # a tensor of ones
t4 = t3.numpy()            # convert: t4 shares data with t3
t3[0] = 50                 # change the tensor in place
print(t4)                  # the array changed too: [50. 1. 1.]

# Q4: clone first so they don't share
t5 = torch.ones(3)         # a tensor of ones
t6 = t5.clone().numpy()    # copy first, then convert
t5[0] = 50                 # change the tensor in place
print(t6)                  # the array is unchanged: [1. 1. 1.]

torch.float32 float32
[0. 0. 0. 0. 0.]
[50.  1.  1.]
[1. 1. 1.]


Reproducibility

- Random numbers in PyTorch come from a fixed list (they are pseudo-random)
- `torch.manual_seed(n)` picks list number n and starts at its beginning
- Each `torch.rand(...)` reads the next numbers from that list
- Setting the same seed again goes back to the start, so you get the same numbers
- Setting the seed only once and calling rand twice gives different numbers (it keeps reading)
- Any whole number works as a seed; 42 is just a popular joke from a novel
- Different seeds give different lists
- `a == b` compares two tensors position by position (True / False)
- Seeds make results repeatable: a model starts with the same random weights every run

Q1. Create two random tensors of shape (2, 2) without a seed. Compare them with ==. Are they equal?

Q2. Set seed 7, then create a random tensor of shape (2, 2). Set seed 7 again, then create another one. Compare them with ==.

Q3. Set seed 7 only once, then create two random (2, 2) tensors back to back. Compare them with ==. Are they equal? Why?

Q4. Set seed 42 and create torch.rand(1) twice. Do you get 0.8823, then 0.9150?

In [8]:
t1 = torch.rand(2,2)
t2 = torch.rand(2,2)
print(t1 == t2)

torch.manual_seed(7)
t3 = torch.rand(2,2)
torch.manual_seed(7)
t4 = torch.rand(2,2)
print(t3 == t4)

torch.manual_seed(7)
t5 = torch.rand(2,2)
t6 = torch.rand(2,2)
print(t5 == t6)

torch.manual_seed(42)
print(torch.rand(1))


tensor([[False, False],
        [False, False]])
tensor([[True, True],
        [True, True]])
tensor([[False, False],
        [False, False]])
tensor([0.8823])


Checking for a GPU

- A GPU does thousands of simple calculations at the same time, which makes deep learning faster
- CUDA is NVIDIA's software for using the GPU; `torch.cuda` is PyTorch's GPU part
- PyTorch's GPU support only works with NVIDIA GPUs (not Intel or AMD)
- `torch.cuda.is_available()` asks "can PyTorch use an NVIDIA GPU?" (True / False)
- `torch.cuda.device_count()` asks "how many NVIDIA GPUs can PyTorch see?"
- `!nvidia-smi` runs NVIDIA's tool to show the GPU's name and memory (error = no NVIDIA GPU)
- `!` at the start of a notebook cell runs a terminal command instead of Python
- Apple Silicon Macs use `torch.backends.mps.is_available()` instead
- Without a GPU, PyTorch uses the CPU: slower, but everything still works

In [11]:
print(torch.cuda.is_available())   # asks: "can you use an NVIDIA GPU?" (True / False)
print(torch.cuda.device_count())   # asks: "how many NVIDIA GPUs can you see?"
!nvidia-smi   # runs NVIDIA's tool that describes the GPU

False
0


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


Free GPU on Google Colab

- Laptops without an NVIDIA GPU can borrow one for free on Google Colab
- Steps: colab.research.google.com -> New notebook -> Runtime -> Change runtime type -> T4 GPU -> Save
- PyTorch is already installed on Colab (no `%pip install` needed)
- The first run is slow: Colab connects to a GPU computer and loads PyTorch's GPU files
- With the GPU on: `torch.cuda.is_available()` is True and `torch.cuda.device_count()` is 1
- `!nvidia-smi` shows the GPU's name (usually Tesla T4) and memory (about 15 GB)
- Switching back to CPU makes `torch.cuda.is_available()` False again
- Changing the runtime type resets the session; rerun cells from the top
- Free GPU time is limited, and sessions reset when idle
- Save Colab notebooks with File -> Save a copy in Drive
- Use the GPU only when needed; for small tensors the CPU is fine

 Choosing the device

- Device-agnostic code runs on CPU or GPU without changing any lines
- `device = "cuda" if torch.cuda.is_available() else "cpu"` picks the GPU if there is one
- Tensors are created on the CPU by default
- `x.to(device)` makes a copy of x on the chosen device
- `.to(device)` returns a new tensor: store it back with `x = x.to(device)`
- A tensor on the GPU prints `device='cuda:0'` (the first NVIDIA GPU)

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"  # use the NVIDIA GPU if there is one, otherwise the CPU
print(device)                                            # show which device was chosen

cpu


In [15]:
x = torch.tensor([1, 2, 3])  # create a tensor (on the CPU by default)
print(x.device)              # where x lives
x = x.to(device)             # move x to the chosen device and store it back
print(x)                     # display it
print(x.device)              # where x lives now

cpu
tensor([1, 2, 3])
cpu


Back to the CPU and the same-device rule

- NumPy (and matplotlib) only work with CPU tensors
- GPU tensor to NumPy: `x.cpu().numpy()` (copy to the CPU first)
- `.cpu()` does nothing on a tensor already on the CPU, so `.cpu().numpy()` works everywhere
- Tensors must be on the same device to be combined
- Mixing devices gives: `Expected all tensors to be on the same device`
- Fix: move every tensor with `.to(device)`
- This is the "wrong device" error: remember "what, what, where" (shape, dtype, device)